# Loan Performance Intelligence Engine — End-to-End Walkthrough

This notebook demonstrates the key outputs from our pipeline. Before running this notebook, ensure you have executed `python run_all.py` at the project root so all reports and data files are generated.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
import json
import os

ROOT_DIR = "../"
REPORTS_DIR = os.path.join(ROOT_DIR, "reports")
DATA_DIR = os.path.join(ROOT_DIR, "data/processed")

## 1. Data Intelligence & Profiling
Let's look at the top validation rule violations detected in the raw data.

In [2]:
try:
    violations = pd.read_csv(os.path.join(DATA_DIR, "validation_violations.csv"))
    display(violations.head())
except FileNotFoundError:
    print("Validation violations file not found. Run python run_all.py first.")

,loan_id,month_index,rule_id,rule_name,severity,details
0,LN0001580,13,VR001,balance_consistency,error,current_balance > 105% of original
1,LN0000655,3,VR001,balance_consistency,error,current_balance > 105% of original
2,LN0004883,12,VR001,balance_consistency,error,current_balance > 105% of original
3,LN0001887,10,VR001,balance_consistency,error,current_balance > 105% of original
4,LN0001140,5,VR001,balance_consistency,error,current_balance > 105% of original


## 2. Time-Aware Split and Leakage Proof
Our splits ensure no data leakage. Let's verify the months in the test set.

In [3]:
try:
    test_data = pd.read_csv(os.path.join(ROOT_DIR, "data/raw/loan_monthly_performance_test.csv"))
    print("Test set months:", test_data["month_index"].unique())
except FileNotFoundError:
    print("Test set not found.")

Test set months: [33 40 34 39 32 27 30 35 26 29 41 31 28 36 38 37 42]


## 3. Model Metrics
A summary of our binary prediction models.

In [4]:
try:
    with open(os.path.join(DATA_DIR, "binary_model_metrics.json")) as f:
        metrics = json.load(f)
    
    df_metrics = pd.DataFrame([
        {"Target": tgt, "Model": model, **res}
        for tgt, models in metrics.items()
        for model, res in models.items() if "test" in model
    ])
    display(df_metrics)
except Exception as e:
    print("Metrics not found:", e)

,Target,Model,target,roc_auc,pr_auc,f1,brier,rec_p80
0,next_3m_delinquency_flag,LR_test,next_3m_delinquency_flag,0.6618,0.4887,0.4089,0.2132,0.0000
1,next_3m_delinquency_flag,LGBM_test,next_3m_delinquency_flag,0.7129,0.5732,0.4604,0.1973,0.0968
2,next_6m_delinquency_flag,LR_test,next_6m_delinquency_flag,0.6580,0.6148,0.2262,0.2688,0.0000
3,next_6m_delinquency_flag,LGBM_test,next_6m_delinquency_flag,0.7098,0.6873,0.6801,0.2161,0.2094
4,next_12m_default_flag,LR_test,next_12m_default_flag,0.6966,0.2743,0.1665,0.1222,0.0000
5,next_12m_default_flag,LGBM_test,next_12m_default_flag,0.6966,0.2743,0.1665,0.1222,0.0000
6,next_12m_prepayment_flag,LR_test,next_12m_prepayment_flag,0.5652,0.3150,0.4377,0.2459,0.0000
7,next_12m_prepayment_flag,LGBM_test,next_12m_prepayment_flag,0.5652,0.3150,0.4377,0.2459,0.0000


## 4. Anomaly Detection
Top flagged exceptions using our hybrid rules+ML approach.

In [5]:
try:
    anomalies = pd.read_csv(os.path.join(DATA_DIR, "anomaly_scores.csv"))
    flagged = anomalies[anomalies["exception_flag"] == 1].sort_values("anomaly_score", ascending=False)
    display(flagged[["loan_id", "anomaly_score", "predicted_exception_type", "top_drivers"]].head(10))
except Exception as e:
    print("Anomalies not found:", e)

,loan_id,anomaly_score,predicted_exception_type,top_drivers
72671,LN0000298,0.678280,DocumentGap,balance_utilisation(z=11.76); default_flag(z=6...
24326,LN0001932,0.666667,DocumentGap,current_balance(z=25.15); balance_utilisation(...
3341,LN0001319,0.657098,DocumentGap,current_balance(z=12.55); balance_utilisation(...
48914,LN0000351,0.638472,DocumentGap,balance_utilisation(z=11.76); default_flag(z=6...
13065,LN0003719,0.623182,Anomaly:IsolationForest,current_balance(z=17.12); balance_utilisation(...
10977,LN0003669,0.618711,Anomaly:IsolationForest,current_balance(z=19.82); balance_utilisation(...
55909,LN0003051,0.617892,DocumentGap,balance_utilisation(z=11.76); current_balance(...
16274,LN0004754,0.617551,Anomaly:IsolationForest,balance_utilisation(z=11.76); current_balance(...
30194,LN0004853,0.617311,Anomaly:IsolationForest,current_balance(z=12.54); balance_utilisation(...
37644,LN0001810,0.612763,Anomaly:IsolationForest,current_balance(z=14.86); balance_utilisation(...


## 5. Final Submission
The output template to submit.

In [6]:
try:
    sub = pd.read_csv(os.path.join(ROOT_DIR, "submission/submission.csv"))
    display(sub.head())
except Exception as e:
    print("Submission not found:", e)

,loan_id,as_of_month,prob_delinquency_3m,prob_delinquency_6m,prob_default_12m,prob_prepayment_12m,predicted_next_state,exception_flag,exception_type,anomaly_score,top_drivers,recommended_action,confidence
0,LN0000004,2021-11,0.276933,0.501182,0.098548,0.551625,Current,0,NaN,0.108480,NaN,No immediate action,0.6613
1,LN0000004,2021-12,0.455796,0.705290,0.118243,0.539940,Current,0,NaN,0.119399,NaN,No immediate action,0.6644
2,LN0000004,2022-01,0.461122,0.692346,0.111183,0.539759,Current,0,NaN,0.135032,NaN,No immediate action,0.6701
3,LN0000004,2022-02,0.465207,0.705290,0.104490,0.539573,Current,0,NaN,0.144693,NaN,No immediate action,0.6624
4,LN0000004,2022-03,0.454601,0.688309,0.098180,0.539412,Current,0,NaN,0.155233,NaN,No immediate action,0.6625
